# 03 — Model Training
**Brugada-HUCA ECG Dataset**

This notebook trains two model families and saves their artifacts:

| Stage | Models |
|---|---|
| **Classical ML** | Logistic Regression (baseline), Random Forest (tuned) |
| **Deep Learning** | 1D ResNet (primary model) |

Both are evaluated on the **validation set** here.  
Final test-set evaluation is in `04_evaluation.ipynb`.

## 0 — Setup

In [ ]:
import sys, pickle
from pathlib import Path

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'src' / 'config.py').exists()),
    _cwd,
)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import config as cfg
from preprocessing import load_processed, compute_pos_weight
from features import extract_batch
from dataset import ECGDataset, make_loaders
from models import ResNet1D
from train import run_training, load_checkpoint
from evaluate import plot_training_curves, compute_metrics, predict_dl

REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
torch.manual_seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

## 1 — Load Processed Data

In [ ]:
X_train, y_train, ids_train = load_processed('train', cfg.DATA_PROCESSED)
X_val,   y_val,   ids_val   = load_processed('val',   cfg.DATA_PROCESSED)
X_test,  y_test,  ids_test  = load_processed('test',  cfg.DATA_PROCESSED)

print(f'Train: {X_train.shape}  Brugada={( y_train==1).sum()}  Normal={(y_train==0).sum()}')
print(f'Val  : {X_val.shape}   Brugada={(y_val==1).sum()}  Normal={(y_val==0).sum()}')
print(f'Test : {X_test.shape}  Brugada={(y_test==1).sum()}  Normal={(y_test==0).sum()}')

---
## Part A — Classical ML
### A1 — Feature Extraction

In [ ]:
# Cache features to disk so re-running is fast
feat_cache = cfg.DATA_PROCESSED / 'features_train.parquet'

if feat_cache.exists():
    print('Loading cached features...')
    df_train_feat = pd.read_parquet(feat_cache)
else:
    print('Extracting features (this may take a few minutes)...')
    df_train_feat = extract_batch(X_train, ids=ids_train)
    df_train_feat.to_parquet(feat_cache)

feat_cache_val = cfg.DATA_PROCESSED / 'features_val.parquet'
if feat_cache_val.exists():
    df_val_feat = pd.read_parquet(feat_cache_val)
else:
    df_val_feat = extract_batch(X_val, ids=ids_val)
    df_val_feat.to_parquet(feat_cache_val)

feat_cache_test = cfg.DATA_PROCESSED / 'features_test.parquet'
if feat_cache_test.exists():
    df_test_feat = pd.read_parquet(feat_cache_test)
else:
    df_test_feat = extract_batch(X_test, ids=ids_test)
    df_test_feat.to_parquet(feat_cache_test)

print(f'\nFeature matrix shape (train): {df_train_feat.shape}')
df_train_feat.head(3)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize features (fit on train only)
scaler = StandardScaler()
X_tr_ml = scaler.fit_transform(df_train_feat.values)
X_vl_ml = scaler.transform(df_val_feat.values)
X_te_ml = scaler.transform(df_test_feat.values)

# Save scaler for inference
with open(cfg.MODELS_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Feature shapes:', X_tr_ml.shape, X_vl_ml.shape, X_te_ml.shape)

### A2 — Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
cw = {0: 1.0, 1: n_neg / n_pos}   # class weights mirror pos_weight

lr = LogisticRegression(
    max_iter=2000, class_weight=cw, solver='lbfgs',
    C=0.1, random_state=cfg.RANDOM_SEED
)

# 5-fold stratified cross-validation on TRAIN set
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.RANDOM_SEED)
cv_auroc_lr = cross_val_score(lr, X_tr_ml, y_train, cv=skf, scoring='roc_auc')
print(f'LogReg CV AUROC: {cv_auroc_lr.mean():.3f} ± {cv_auroc_lr.std():.3f}')

# Fit on full train set
lr.fit(X_tr_ml, y_train)

# Save
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(cfg.MODELS_DIR / 'lr_best.pkl', 'wb') as f:
    pickle.dump(lr, f)

# Validate
lr_prob_val = lr.predict_proba(X_vl_ml)[:, 1]
lr_metrics_val = compute_metrics(y_val, lr_prob_val)
print('\nLogReg — Val metrics:')
for k, v in lr_metrics_val.items():
    print(f'  {k}: {v}')

### A3 — Random Forest + GridSearchCV

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 400],
    'max_depth':    [None, 15, 25],
    'min_samples_leaf': [1, 2],
}

rf_base = RandomForestClassifier(
    class_weight='balanced', random_state=cfg.RANDOM_SEED, n_jobs=-1
)

gs = GridSearchCV(
    rf_base, param_grid, cv=skf, scoring='roc_auc',
    n_jobs=-1, verbose=1, refit=True
)
gs.fit(X_tr_ml, y_train)

print(f'\nBest RF params: {gs.best_params_}')
print(f'Best CV AUROC : {gs.best_score_:.3f}')

rf = gs.best_estimator_

# Save RF
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(cfg.MODELS_DIR / 'rf_best.pkl', 'wb') as f:
    pickle.dump(rf, f)

rf_prob_val = rf.predict_proba(X_vl_ml)[:, 1]
rf_metrics_val = compute_metrics(y_val, rf_prob_val)
print('\nRandom Forest — Val metrics:')
for k, v in rf_metrics_val.items():
    print(f'  {k}: {v}')

### A4 — Feature Importance (top 20)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=df_train_feat.columns)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
top20[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig(REPORTS / 'rf_feature_importance.png', dpi=150)
plt.show()

---
## Part B — Deep Learning: 1D ResNet
### B1 — DataLoaders

In [ ]:
splits = {
    'X_train': X_train, 'y_train': y_train, 'ids_train': ids_train,
    'X_val':   X_val,   'y_val':   y_val,   'ids_val':   ids_val,
    'X_test':  X_test,  'y_test':  y_test,  'ids_test':  ids_test,
}

loaders = make_loaders(splits, batch_size=cfg.BATCH_SIZE)

print('DataLoader sizes:')
for name, dl in loaders.items():
    print(f'  {name}: {len(dl.dataset)} samples, {len(dl)} batches')

### B2 — Model Architecture

In [ ]:
model = ResNet1D(
    n_leads=cfg.N_LEADS,
    base_channels=64,
    kernel_size=7,
    dropout=0.2,
).to(DEVICE)

print(f'Parameters: {model.count_parameters():,}')

# Dry run to verify output shape
with torch.no_grad():
    dummy = torch.zeros(2, cfg.N_LEADS, cfg.N_SAMPLES).to(DEVICE)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  (expected: [2, 1])')

### B3 — Loss, Optimizer, Scheduler

In [ ]:
pos_w = compute_pos_weight(y_train)
print(f'pos_weight = {pos_w:.2f}')

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_w, dtype=torch.float32).to(DEVICE)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-5
)

print('Optimizer:', optimizer)
print('Scheduler: CosineAnnealingLR')

### B4 — Training Loop

In [ ]:
CHECKPOINT = cfg.MODELS_DIR / 'best_resnet1d.pt'

history = run_training(
    model         = model,
    train_loader  = loaders['train'],
    val_loader    = loaders['val'],
    optimizer     = optimizer,
    criterion     = criterion,
    scheduler     = scheduler,
    device        = DEVICE,
    num_epochs    = cfg.NUM_EPOCHS,
    patience      = cfg.PATIENCE,
    checkpoint_path = CHECKPOINT,
    verbose       = True,
)

print(f'\nBest val AUROC: {max(history["val_auroc"]):.4f} at epoch {int(np.argmax(history["val_auroc"]))+1}')

### B5 — Training Curves

In [ ]:
from evaluate import plot_training_curves

plot_training_curves(history, save_path=REPORTS / 'training_curves.png')

### B6 — Reload Best Checkpoint and Validate

In [ ]:
best_epoch = load_checkpoint(CHECKPOINT, model)
print(f'Loaded best checkpoint from epoch {best_epoch}')

y_true_val, y_prob_val_dl = predict_dl(model, loaders['val'], device=DEVICE)
dl_metrics_val = compute_metrics(y_true_val, y_prob_val_dl)

print('\n1D ResNet — Val metrics:')
for k, v in dl_metrics_val.items():
    print(f'  {k}: {v}')

## Summary — Validation Comparison

Quick look at key metrics before full test evaluation.

In [ ]:
from evaluate import compare_models

val_results = {
    'LogReg':    lr_metrics_val,
    'RandomForest': rf_metrics_val,
    'ResNet1D':  dl_metrics_val,
}

df_val_cmp = compare_models(val_results)
print('\nValidation Set — Model Comparison:')
display(df_val_cmp)

In [ ]:
# Save val probabilities for notebook 04
import pickle
val_preds = {
    'LogReg':       {'y_true': y_val,      'y_prob': lr_prob_val},
    'RandomForest': {'y_true': y_val,      'y_prob': rf_prob_val},
    'ResNet1D':     {'y_true': y_true_val, 'y_prob': y_prob_val_dl},
}
with open(cfg.MODELS_DIR / 'val_predictions.pkl', 'wb') as f:
    pickle.dump(val_preds, f)

print('Saved val predictions to models/val_predictions.pkl')
print('Proceed to 04_evaluation.ipynb for test-set evaluation.')